# Traffic Light Detection (YOLOv8n) - Colab Training + Smoke Test

Single-model detector where class encodes state: `redlight`, `yellowlight`, `greenlight`.

This notebook is optimized for Google Colab GPU training and includes:
- dataset validation
- 1-epoch smoke test (CUDA)
- full training cell
- evaluation
- inference for image/folder/video with annotated outputs
- temporal smoothing (`NONE`, `RED`, `YELLOW`, `GREEN`)


In [11]:
%pip install -q ultralytics opencv-python pyyaml

import ultralytics
import torch

print('ultralytics:', ultralytics.__version__)
print('torch:', torch.__version__)


ultralytics: 8.4.19
torch: 2.10.0+cu128


## GPU Training Optimization Tips

**For best results on Colab:**
1. **GPU Selection**: Use `Runtime > Change runtime type > T4 GPU` (free) or `A100` (Colab Pro)
2. **Batch Size**: Set `BATCH = -1` for auto-batch optimization (maximizes GPU memory)
3. **Image Caching**: `cache='ram'` loads images into RAM for faster epoch iterations
4. **Mixed Precision**: `amp=True` enables FP16 training (2x faster, less memory)
5. **Model Choice**: Start with `yolov8s.pt` for better accuracy than nano with acceptable speed

**Colab session tips:**
- Keep the tab active to prevent disconnection
- Use `save_period=10` to checkpoint every 10 epochs
- Best weights auto-copy to Drive after training


In [12]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA GPU not available. In Colab: Runtime > Change runtime type > GPU')


CUDA available: True
GPU: Tesla T4


In [23]:
from google.colab import drive
from pathlib import Path
import yaml

drive.mount('/content/drive')

# Update this path if your dataset lives elsewhere in Drive.
DATASET_ROOT = Path('/content/drive/MyDrive/traffic_lights')
DATA_YAML_PATH = DATASET_ROOT / 'data.yaml'

print('DATASET_ROOT:', DATASET_ROOT)
print('DATA_YAML_PATH:', DATA_YAML_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET_ROOT: /content/drive/MyDrive/traffic_lights
DATA_YAML_PATH: /content/drive/MyDrive/traffic_lights/data.yaml


In [26]:
from pathlib import Path

base = Path("/content/drive/MyDrive")
candidates = []

for p in base.glob("**/images/train"):
    root = p.parent.parent  # .../<root>/images/train -> <root>
    if (root / "images/val").exists() and (root / "labels/train").exists() and (root / "labels/val").exists():
        candidates.append(root)

print("Valid dataset roots found:")
for i, c in enumerate(candidates):
    print(f"[{i}] {c}")

if not candidates:
    raise RuntimeError("No valid dataset root found in MyDrive with images/train,val and labels/train,val")


Valid dataset roots found:


RuntimeError: No valid dataset root found in MyDrive with images/train,val and labels/train,val

In [18]:
required_dirs = [
    DATASET_ROOT / 'images' / 'train',
    DATASET_ROOT / 'images' / 'val',
    DATASET_ROOT / 'labels' / 'train',
    DATASET_ROOT / 'labels' / 'val',
]

for d in required_dirs:
    if not d.exists():
        raise FileNotFoundError(f'Missing required directory: {d}')

yaml_payload = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'redlight', 1: 'yellowlight', 2: 'greenlight'},
}

with open(DATA_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(yaml_payload, f, sort_keys=False)

print(DATA_YAML_PATH.read_text(encoding='utf-8'))


FileNotFoundError: Missing required directory: /content/drive/MyDrive/<YOUR_REAL_FOLDER>/images/train

In [ ]:
# Dataset validation: missing labels/images, invalid class IDs, malformed YOLO boxes.
from collections import defaultdict

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def _image_files(path):
    return sorted([p for p in path.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES])

def _validate_split(dataset_root, split):
    errors = defaultdict(list)
    img_dir = dataset_root / 'images' / split
    lbl_dir = dataset_root / 'labels' / split

    image_files = _image_files(img_dir)
    label_files = sorted([p for p in lbl_dir.iterdir() if p.is_file() and p.suffix.lower() == '.txt'])

    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in label_files}

    for stem in sorted(image_stems - label_stems):
        errors['missing_labels'].append(f'{split}: image {stem} missing label')
    for stem in sorted(label_stems - image_stems):
        errors['missing_images'].append(f'{split}: label {stem}.txt missing image')

    for label_path in label_files:
        lines = label_path.read_text(encoding='utf-8').splitlines()
        for idx, raw in enumerate(lines, start=1):
            line = raw.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                errors['malformed_rows'].append(f'{label_path}:{idx} expected 5 values')
                continue
            cls_raw, x_raw, y_raw, w_raw, h_raw = parts
            try:
                cls_id = int(cls_raw)
            except ValueError:
                errors['invalid_class_ids'].append(f'{label_path}:{idx} class is not int: {cls_raw}')
                continue
            if cls_id not in (0, 1, 2):
                errors['invalid_class_ids'].append(f'{label_path}:{idx} class out of range: {cls_id}')

            try:
                x, y, w, h = map(float, (x_raw, y_raw, w_raw, h_raw))
            except ValueError:
                errors['malformed_rows'].append(f'{label_path}:{idx} bbox values not float')
                continue

            if not (0 <= x <= 1 and 0 <= y <= 1 and 0 <= w <= 1 and 0 <= h <= 1):
                errors['malformed_boxes'].append(f'{label_path}:{idx} values outside [0,1]: {(x,y,w,h)}')
            if w <= 0 or h <= 0:
                errors['malformed_boxes'].append(f'{label_path}:{idx} width/height must be > 0: {(w,h)}')
            if x - w/2 < 0 or x + w/2 > 1 or y - h/2 < 0 or y + h/2 > 1:
                errors['malformed_boxes'].append(f'{label_path}:{idx} box extends outside image: {(x,y,w,h)}')

    return errors

all_errors = defaultdict(list)
for split_name in ['train', 'val']:
    split_errors = _validate_split(DATASET_ROOT, split_name)
    for k, vals in split_errors.items():
        all_errors[k].extend(vals)

total_errors = sum(len(v) for v in all_errors.values())
print('Validation summary:')
for k in sorted(all_errors.keys()):
    print(f'  {k}: {len(all_errors[k])}')
print('  total_errors:', total_errors)

MAX_PRINT = 20
for k in sorted(all_errors.keys()):
    for msg in all_errors[k][:MAX_PRINT]:
        print('   -', msg)

if total_errors > 0:
    raise RuntimeError('Dataset validation failed. Fix dataset issues before training.')

print('Dataset validation PASS')


In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch

# ============ MODEL SELECTION ============
# Available pretrained models (ranked by size/accuracy tradeoff):
# - yolov8n.pt: Nano (fastest, ~3.2M params) - good for quick experiments
# - yolov8s.pt: Small (~11.2M params) - better accuracy, still fast
# - yolov8m.pt: Medium (~25.9M params) - balanced choice for production
# - yolov8l.pt: Large (~43.7M params) - high accuracy
# - yolov8x.pt: Extra Large (~68.2M params) - best accuracy, slowest
MODEL_CANDIDATES = ['yolov8s.pt', 'yolov8n.pt', 'yolo8n.pt']

# ============ TRAINING HYPERPARAMETERS ============
IMGSZ = 640                    # Image size (640 is standard; use 1280 if small objects)
BATCH = -1                     # -1 = auto-batch (maximizes GPU memory usage)
SMOKE_EPOCHS = 1               # Quick validation
FULL_TRAIN_EPOCHS = 100        # More epochs for better convergence
CONF_THRESHOLD = 0.35          # Inference confidence threshold

# ============ OPTIMIZER SETTINGS ============
OPTIMIZER = 'AdamW'            # AdamW typically works best for YOLO
LR0 = 0.01                     # Initial learning rate
LRF = 0.01                     # Final LR = LR0 * LRF (cosine decay target)
MOMENTUM = 0.937               # SGD momentum / Adam beta1
WEIGHT_DECAY = 0.0005          # L2 regularization
WARMUP_EPOCHS = 3.0            # Warmup epochs
WARMUP_MOMENTUM = 0.8          # Warmup initial momentum
WARMUP_BIAS_LR = 0.1           # Warmup initial bias LR

# ============ AUGMENTATION SETTINGS ============
# Data augmentation (tuned for traffic light detection)
HSV_H = 0.015                  # HSV-Hue augmentation (important for color detection)
HSV_S = 0.7                    # HSV-Saturation augmentation
HSV_V = 0.4                    # HSV-Value augmentation (lighting robustness)
DEGREES = 0.0                  # Rotation (keep 0 - traffic lights are upright)
TRANSLATE = 0.1                # Translation (fraction)
SCALE = 0.5                    # Scale augmentation (gain)
SHEAR = 0.0                    # Shear (keep 0 for traffic lights)
PERSPECTIVE = 0.0              # Perspective (keep 0 for traffic lights)
FLIPUD = 0.0                   # Vertical flip (keep 0 - traffic lights don't flip)
FLIPLR = 0.5                   # Horizontal flip probability
MOSAIC = 1.0                   # Mosaic augmentation probability
MIXUP = 0.1                    # MixUp augmentation probability
COPY_PASTE = 0.1               # Copy-paste augmentation probability

# ============ REGULARIZATION ============
DROPOUT = 0.0                  # Dropout rate (use 0.1-0.3 if overfitting)
LABEL_SMOOTHING = 0.0          # Label smoothing epsilon

# ============ LOSS WEIGHTS ============
BOX = 7.5                      # Box loss weight
CLS = 0.5                      # Class loss weight
DFL = 1.5                      # DFL loss weight

# ============ EARLY STOPPING ============
PATIENCE = 30                  # Early stopping patience (epochs)

# ============ OUTPUT DIRECTORIES ============
PROJECT_DIR = '/content/runs/tl'
SMOKE_RUN_NAME = 'smoke_yv8s'
FULL_RUN_NAME = 'full_yv8s'

# ============ GPU OPTIMIZATION ============
# Enable TF32 for faster training on Ampere+ GPUs (A100, T4, etc.)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True  # Enable cuDNN autotuner

# ============ MODEL LOADING ============
selected_model = None
model = None
for candidate in MODEL_CANDIDATES:
    try:
        model = YOLO(candidate)
        selected_model = candidate
        print('Using model candidate:', selected_model)
        break
    except Exception as exc:
        print('Failed model candidate', candidate, '->', exc)

if model is None:
    raise RuntimeError('Could not initialize YOLO model from candidates')

# Print GPU memory info
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory: {gpu_mem_gb:.1f} GB')
    if gpu_mem_gb >= 15:
        print('High-memory GPU detected - using batch=-1 for auto-batch')


In [ ]:
# Smoke test: 1-epoch CUDA train with full augmentation pipeline.
import gc

# Clear GPU memory before training
gc.collect()
torch.cuda.empty_cache()

try:
    smoke_results = model.train(
        data=str(DATA_YAML_PATH),
        imgsz=IMGSZ,
        batch=BATCH,
        epochs=SMOKE_EPOCHS,
        device=0,
        project=PROJECT_DIR,
        name=SMOKE_RUN_NAME,
        # Optimizer
        optimizer=OPTIMIZER,
        lr0=LR0,
        lrf=LRF,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        warmup_epochs=WARMUP_EPOCHS,
        warmup_momentum=WARMUP_MOMENTUM,
        warmup_bias_lr=WARMUP_BIAS_LR,
        # Augmentation
        hsv_h=HSV_H,
        hsv_s=HSV_S,
        hsv_v=HSV_V,
        degrees=DEGREES,
        translate=TRANSLATE,
        scale=SCALE,
        shear=SHEAR,
        perspective=PERSPECTIVE,
        flipud=FLIPUD,
        fliplr=FLIPLR,
        mosaic=MOSAIC,
        mixup=MIXUP,
        copy_paste=COPY_PASTE,
        # Regularization
        dropout=DROPOUT,
        label_smoothing=LABEL_SMOOTHING,
        # Loss weights
        box=BOX,
        cls=CLS,
        dfl=DFL,
        # Performance
        workers=4,                   # Dataloader workers (increase for faster loading)
        cache='ram',                 # Cache images in RAM for faster training
        amp=True,                    # Automatic Mixed Precision (FP16) for speed
        patience=PATIENCE,
        verbose=True,
    )
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower():
        gc.collect()
        torch.cuda.empty_cache()
        retry_batch = 8  # Fallback batch size
        print(f'CUDA OOM - retrying with batch={retry_batch}, cache=False')
        smoke_results = model.train(
            data=str(DATA_YAML_PATH),
            imgsz=IMGSZ,
            batch=retry_batch,
            epochs=SMOKE_EPOCHS,
            device=0,
            project=PROJECT_DIR,
            name=SMOKE_RUN_NAME + '_oom_retry',
            optimizer=OPTIMIZER,
            lr0=LR0,
            lrf=LRF,
            workers=2,
            cache=False,             # Disable cache on OOM
            amp=True,
            patience=PATIENCE,
        )
    else:
        raise

SMOKE_SAVE_DIR = Path(smoke_results.save_dir)
SMOKE_BEST = SMOKE_SAVE_DIR / 'weights' / 'best.pt'

print('Smoke run dir:', SMOKE_SAVE_DIR)
print('Smoke best weights:', SMOKE_BEST)
if not SMOKE_BEST.exists():
    raise RuntimeError('Smoke test failed: best.pt was not created')


In [ ]:
# Smoke test verification inference on one validation image.
val_candidates = sorted((DATASET_ROOT / 'images' / 'val').glob('*'))
val_candidates = [p for p in val_candidates if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}]
if not val_candidates:
    raise RuntimeError('No validation images found for smoke inference test')

test_image = val_candidates[0]
smoke_model = YOLO(str(SMOKE_BEST))
smoke_result = smoke_model.predict(
    source=str(test_image),
    conf=CONF_THRESHOLD,
    imgsz=IMGSZ,
    device=0,
    save=False,
    verbose=False,
)[0]

print('SMOKE TEST: PASS')
print('Image:', test_image)
boxes = smoke_result.boxes
if boxes is None or len(boxes) == 0:
    print('No detections on smoke image (pipeline still valid).')
else:
    for box in boxes:
        cls_id = int(box.cls.item())
        conf = float(box.conf.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        name = smoke_result.names.get(cls_id, str(cls_id))
        print(f'class={name} conf={conf:.3f} bbox=[{x1:.1f},{y1:.1f},{x2:.1f},{y2:.1f}]')


In [ ]:
# Full training (set RUN_FULL_TRAIN = True to run).
RUN_FULL_TRAIN = False

if RUN_FULL_TRAIN:
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    full_model = YOLO(selected_model)
    full_results = full_model.train(
        data=str(DATA_YAML_PATH),
        imgsz=IMGSZ,
        batch=BATCH,
        epochs=FULL_TRAIN_EPOCHS,
        device=0,
        project=PROJECT_DIR,
        name=FULL_RUN_NAME,
        # Optimizer
        optimizer=OPTIMIZER,
        lr0=LR0,
        lrf=LRF,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
        warmup_epochs=WARMUP_EPOCHS,
        warmup_momentum=WARMUP_MOMENTUM,
        warmup_bias_lr=WARMUP_BIAS_LR,
        # Augmentation
        hsv_h=HSV_H,
        hsv_s=HSV_S,
        hsv_v=HSV_V,
        degrees=DEGREES,
        translate=TRANSLATE,
        scale=SCALE,
        shear=SHEAR,
        perspective=PERSPECTIVE,
        flipud=FLIPUD,
        fliplr=FLIPLR,
        mosaic=MOSAIC,
        mixup=MIXUP,
        copy_paste=COPY_PASTE,
        # Regularization
        dropout=DROPOUT,
        label_smoothing=LABEL_SMOOTHING,
        # Loss weights
        box=BOX,
        cls=CLS,
        dfl=DFL,
        # Performance
        workers=4,
        cache='ram',                 # Cache images in RAM (faster training)
        amp=True,                    # Mixed precision (FP16)
        patience=PATIENCE,
        # Saving
        save=True,
        save_period=10,              # Save checkpoint every N epochs
        plots=True,                  # Generate training plots
        verbose=True,
    )
    FULL_SAVE_DIR = Path(full_results.save_dir)
    FULL_BEST = FULL_SAVE_DIR / 'weights' / 'best.pt'
    print('Full train best:', FULL_BEST)

    # Copy best weights to Drive for persistence
    import shutil
    drive_weights_dir = DATASET_ROOT / 'trained_weights'
    drive_weights_dir.mkdir(exist_ok=True)
    shutil.copy(FULL_BEST, drive_weights_dir / 'best.pt')
    print(f'Best weights copied to: {drive_weights_dir / "best.pt"}')
else:
    print('Skipping full training. Set RUN_FULL_TRAIN=True to run full training.')


In [ ]:
# Validation metrics for chosen weights.
weights_for_eval = str(SMOKE_BEST)
if 'FULL_BEST' in globals():
    weights_for_eval = str(FULL_BEST)

eval_model = YOLO(weights_for_eval)
metrics = eval_model.val(data=str(DATA_YAML_PATH), imgsz=IMGSZ, batch=BATCH, device=0)

print('Validation metrics:')
if hasattr(metrics, 'results_dict') and isinstance(metrics.results_dict, dict):
    for k in sorted(metrics.results_dict.keys()):
        print(f'  {k}: {metrics.results_dict[k]}')
else:
    print('  Metrics object format differs by Ultralytics version.')


In [ ]:
# Inference helpers: image, folder, video + temporal smoothing + annotated saves.
from collections import deque
from enum import Enum
import cv2

CLASS_NAMES = ['redlight', 'yellowlight', 'greenlight']
CLASS_COLORS = {
    'redlight': (0, 0, 255),
    'yellowlight': (0, 255, 255),
    'greenlight': (0, 255, 0),
}

class StableState(str, Enum):
    NONE = 'NONE'
    RED = 'RED'
    YELLOW = 'YELLOW'
    GREEN = 'GREEN'

RAW_TO_STABLE = {
    None: StableState.NONE,
    'redlight': StableState.RED,
    'yellowlight': StableState.YELLOW,
    'greenlight': StableState.GREEN,
}

class TemporalStateFilter:
    def __init__(self, window_size=5, min_consecutive=3):
        if min_consecutive > window_size:
            raise ValueError('min_consecutive cannot exceed window_size')
        self.window_size = window_size
        self.min_consecutive = min_consecutive
        self.history = deque(maxlen=window_size)
        self.stable = StableState.NONE

    def update(self, raw_cls):
        state = RAW_TO_STABLE.get(raw_cls, StableState.NONE)
        self.history.append(state)
        if len(self.history) >= self.min_consecutive:
            tail = list(self.history)[-self.min_consecutive:]
            if all(x == tail[0] for x in tail):
                self.stable = tail[0]
        return self.stable

def parse_roi(roi_text):
    if not roi_text:
        return None
    x1, y1, x2, y2 = map(float, roi_text.split(','))
    if not (0 <= x1 < x2 <= 1 and 0 <= y1 < y2 <= 1):
        raise ValueError('ROI must satisfy 0 <= x1 < x2 <= 1 and 0 <= y1 < y2 <= 1')
    return (x1, y1, x2, y2)

def detect_frame(model, frame, conf=0.35, imgsz=640, device=0, roi_norm=None, best_only=True):
    h, w = frame.shape[:2]
    x_off = y_off = 0
    roi_px = None
    infer = frame

    if roi_norm is not None:
        rx1 = int(roi_norm[0] * w)
        ry1 = int(roi_norm[1] * h)
        rx2 = int(roi_norm[2] * w)
        ry2 = int(roi_norm[3] * h)
        roi_px = (rx1, ry1, rx2, ry2)
        infer = frame[ry1:ry2, rx1:rx2]
        x_off, y_off = rx1, ry1

    result = model.predict(source=infer, conf=conf, imgsz=imgsz, device=device, save=False, verbose=False)[0]
    detections = []

    if result.boxes is not None:
        for box in result.boxes:
            cls_id = int(box.cls.item())
            if cls_id not in (0, 1, 2):
                continue
            cls_name = result.names.get(cls_id, str(cls_id))
            score = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            detections.append({
                'class': cls_name,
                'conf': score,
                'bbox': [x1 + x_off, y1 + y_off, x2 + x_off, y2 + y_off],
            })

    detections.sort(key=lambda d: d['conf'], reverse=True)
    if best_only and detections:
        detections = [detections[0]]

    return detections, roi_px

def annotate(frame, detections, stable_state, roi_px=None):
    out = frame.copy()
    if roi_px is not None:
        x1, y1, x2, y2 = roi_px
        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 128, 0), 2)

    for det in detections:
        x1, y1, x2, y2 = [int(v) for v in det['bbox']]
        color = CLASS_COLORS.get(det['class'], (255, 255, 255))
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, f"{det['class']} {det['conf']:.2f}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

    cv2.putText(out, f'Stable: {stable_state.value}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return out

def run_image(model, image_path, output_path, conf=0.35, imgsz=640, device=0, best_only=True, roi_norm=None, smoother=None):
    frame = cv2.imread(str(image_path))
    detections, roi_px = detect_frame(model, frame, conf=conf, imgsz=imgsz, device=device, roi_norm=roi_norm, best_only=best_only)
    raw_cls = detections[0]['class'] if detections else None
    stable = smoother.update(raw_cls) if smoother else StableState.NONE
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        print(f"class={det['class']} conf={det['conf']:.3f} bbox=[{x1:.1f},{y1:.1f},{x2:.1f},{y2:.1f}] stable={stable.value}")
    if not detections:
        print(f'No detections | stable={stable.value}')

    ann = annotate(frame, detections, stable, roi_px=roi_px)
    cv2.imwrite(str(output_path), ann)

def run_folder(model, folder_path, output_dir, conf=0.35, imgsz=640, device=0, best_only=True, roi_norm=None, smoother=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    for p in sorted(folder_path.iterdir()):
        if p.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}:
            continue
        print(f'\n[{p.name}]')
        run_image(model, p, output_dir / p.name, conf=conf, imgsz=imgsz, device=device, best_only=best_only, roi_norm=roi_norm, smoother=smoother)

def run_video(model, video_path, output_path, conf=0.35, imgsz=640, device=0, best_only=True, roi_norm=None, smoother=None):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        detections, roi_px = detect_frame(model, frame, conf=conf, imgsz=imgsz, device=device, roi_norm=roi_norm, best_only=best_only)
        raw_cls = detections[0]['class'] if detections else None
        stable = smoother.update(raw_cls) if smoother else StableState.NONE

        if detections:
            d = detections[0]
            x1, y1, x2, y2 = d['bbox']
            print(f'frame={idx} class={d["class"]} conf={d["conf"]:.3f} bbox=[{x1:.1f},{y1:.1f},{x2:.1f},{y2:.1f}] stable={stable.value}')
        else:
            print(f'frame={idx} no detections stable={stable.value}')

        writer.write(annotate(frame, detections, stable, roi_px=roi_px))
        idx += 1

    cap.release()
    writer.release()
    print('Saved video:', output_path)


In [ ]:
# Example inference calls
from pathlib import Path

weights_for_infer = str(SMOKE_BEST)
if 'FULL_BEST' in globals():
    weights_for_infer = str(FULL_BEST)

infer_model = YOLO(weights_for_infer)
smoother = TemporalStateFilter(window_size=5, min_consecutive=3)

OUTPUT_DIR = Path('/content/infer_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SINGLE_IMAGE = test_image
FOLDER_SOURCE = DATASET_ROOT / 'images' / 'val'
VIDEO_SOURCE = None  # set to Path('/content/drive/MyDrive/path/to/video.mp4') for video inference
ROI_NORM = parse_roi('0,0,1,0.6')  # set to None to disable ROI

print('Running single-image inference...')
run_image(
    infer_model,
    image_path=SINGLE_IMAGE,
    output_path=OUTPUT_DIR / 'single_annotated.jpg',
    conf=CONF_THRESHOLD,
    imgsz=IMGSZ,
    device=0,
    best_only=True,
    roi_norm=ROI_NORM,
    smoother=smoother,
)

print('\nRunning folder inference...')
run_folder(
    infer_model,
    folder_path=FOLDER_SOURCE,
    output_dir=OUTPUT_DIR / 'folder',
    conf=CONF_THRESHOLD,
    imgsz=IMGSZ,
    device=0,
    best_only=True,
    roi_norm=ROI_NORM,
    smoother=smoother,
)

if VIDEO_SOURCE is not None:
    print('\nRunning video inference...')
    smoother_video = TemporalStateFilter(window_size=5, min_consecutive=3)
    run_video(
        infer_model,
        video_path=VIDEO_SOURCE,
        output_path=OUTPUT_DIR / 'video_annotated.mp4',
        conf=CONF_THRESHOLD,
        imgsz=IMGSZ,
        device=0,
        best_only=True,
        roi_norm=ROI_NORM,
        smoother=smoother_video,
    )

print('Outputs saved under:', OUTPUT_DIR)


In [ ]:
# Export best model for deployment.
export_model = YOLO(weights_for_eval)

# ONNX export (universal format)
onnx_path = export_model.export(format='onnx', simplify=True, dynamic=False, imgsz=IMGSZ)
print('Exported ONNX:', onnx_path)

# TensorRT export (fastest inference on NVIDIA GPUs) - optional
EXPORT_TENSORRT = False  # Set True if you have TensorRT installed
if EXPORT_TENSORRT:
    try:
        trt_path = export_model.export(format='engine', device=0, half=True, imgsz=IMGSZ)
        print('Exported TensorRT:', trt_path)
    except Exception as e:
        print(f'TensorRT export failed (optional): {e}')

print('\nIntegration notes:')
print('- ONNX: Universal format, works with ONNX Runtime, OpenCV DNN, etc.')
print('- TensorRT: Fastest on NVIDIA GPUs (2-4x faster than ONNX)')
print('- Detection output: RED -> stop, YELLOW -> caution, GREEN -> continue')
print('- Use temporal smoothing for stable state transitions')
